# Titulo

### **Contexto**
El sistema de transporte público Red Metropolitana de Movilidad en Santiago es una red dinámica que debe adaptar su oferta para satisfacer las variaciones extremas de la demanda a lo largo de la semana. Nuestro proyecto se enfocará en analizar críticamente cómo esta adaptación de la oferta se plasma en los datos de la GTFS (General Transit Feed Specification) vigente y cómo impacta en diferentes áreas geográficas de la ciudad.

El desafío de la planificación no solo radica en cubrir la demanda máxima de los días laborales (peak), sino también en asignar recursos de manera eficiente durante los fines de semana, cuando los patrones de viaje cambian drásticamente (viajes recreativos, menor densidad de pasajeros, horarios de servicio más restringidos).

### **Motivación**
Como estudiantes de ciencia de datos, nos motiva aplicar técnicas de Análisis de Datos para evaluar la equidad y eficiencia del sistema de transporte en función del tiempo y el espacio. Específicamente, buscamos:

**1.- Cuantificar la Disparidad Semanal:** Mapear y cuantificar las diferencias en la oferta de transporte programada (frecuencias, cobertura de paradas) entre un día laboral y un fin de semana. Esto es crucial, ya que la diferencia en la calidad del servicio entre semana y fin de semana afecta la movilidad de los trabajadores con horarios no tradicionales y el acceso a servicios básicos.

**2.- Identificar la Vulnerabilidad Geográfica:** Analizar si las comunas periféricas o sectores específicos, que dependen fuertemente del bus, experimentan una reducción de servicio proporcionalmente mayor durante el fin de semana en comparación con las zonas centrales. Este análisis espacial tiene implicancias directas en la planificación urbana.

**3.- Generar decisiones operacionales:** Nuestro objetivo es generar visualizaciones y métricas concisas que puedan ser utilizadas por la DTPM o futuros estudios para optimizar la programación de itinerarios (frecuencias y distribución de flota) en función de la localización y el día, buscando la estabilidad y predictibilidad del servicio.


In [1]:
import datetime

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib as plt

Chat: 

- Deberiamos incluir en el notebook que no encontramos datos especificos y explicitar que hicimos ¿que les parece?

-¿quien hace le grafica?



In [ ]:
# files = [
#     "../data/reduced/2025-04-21.csv",
#     "../data/reduced/2025-04-22.csv",
#     "../data/reduced/2025-04-23.csv",
#     "../data/reduced/2025-04-27.csv",
#     "../data/reduced/2025-04-24.csv",
#     "../data/reduced/2025-04-25.csv",
#     "../data/reduced/2025-04-26.csv"
# ]

# files = [
#     "../data/sample/2025-04-21_sample.csv",
#     "../data/sample/2025-04-22_sample.csv",
#     "../data/sample/2025-04-23_sample.csv",
#     "../data/sample/2025-04-27_sample.csv",
#     "../data/sample/2025-04-24_sample.csv",
#     "../data/sample/2025-04-25_sample.csv",
#     "../data/sample/2025-04-26_sample.csv"
# ]

# dfs = []
# for file in files:
#     df = pd.read_csv(file)
#     dfs.append(df)


In [ ]:
# df = dfs[0]
df = pd.read_csv("../data/sample/2025-04-21.csv")
df.head()

,tipo_transporte,tiene_bajada,tiempo_subida,tiempo_bajada,tiempo_etapa,comuna_subida,comuna_bajada,parada_subida,parada_bajada,dist_ruta_paraderos,dist_eucl_paraderos,x_subida,y_subida,x_bajada,y_bajada
0,BUS,1,2025-04-21 08:48:04,2025-04-21 08:50:39,155,RECOLETA,RECOLETA,T-4-19-SN-40,E-4-19-SN-55,853,825,347180,6301636,347201,6302489
1,BUS,1,2025-04-21 08:51:46,2025-04-21 08:54:58,192,RECOLETA,RECOLETA,E-4-19-SN-55,L-4-4-50-OP,1090,983,347200,6302473,346625,6303299
2,BUS,1,2025-04-21 15:34:28,2025-04-21 15:39:01,273,RECOLETA,RECOLETA,L-4-12-20-PO,E-4-295-OP-5,1127,959,346563,6303315,347168,6302522
3,METRO,0,2025-04-21 17:27:55,-,-,LAS CONDES,-,LOS DOMINICOS,-,-,-,356329,6302427,-,-
4,METRO,1,2025-04-21 09:12:16,2025-04-21 09:38:40,1584,ESTACION CENTRAL,SANTIAGO,ESTACION CENTRAL,PLAZA DE ARMAS,6240,3064,343933,6297455,346372,6299031


In [ ]:
df.replace("-", np.nan, inplace=True)

In [ ]:
transfo_df = df.copy()
transfo_df["tiempo_subida"] = pd.to_datetime(transfo_df["tiempo_subida"])
transfo_df["tiempo_bajada"] = pd.to_datetime(transfo_df["tiempo_bajada"])

transfo_df["tipo_transporte"] = transfo_df["tipo_transporte"].astype("category")
transfo_df["comuna_subida"] = transfo_df["comuna_subida"].astype("category")
transfo_df["comuna_bajada"] = transfo_df["comuna_bajada"].astype("category")
transfo_df["parada_bajada"] = transfo_df["parada_bajada"].astype("category")
transfo_df["parada_subida"] = transfo_df["parada_subida"].astype("category")

transfo_df["tiempo_etapa"] = transfo_df["tiempo_etapa"].astype(float)
transfo_df["x_bajada"] = transfo_df["x_bajada"].astype(float)
transfo_df["y_bajada"] = transfo_df["y_bajada"].astype(float)
transfo_df["x_subida"] = transfo_df["x_bajada"].astype(float)
transfo_df["y_subida"] = transfo_df["y_bajada"].astype(float)
transfo_df["dist_eucl_paraderos"] = transfo_df["dist_eucl_paraderos"].astype(float)
transfo_df["dist_ruta_paraderos"] = transfo_df["dist_ruta_paraderos"].astype(float)

transfo_df["tiene_bajada"] = transfo_df["tiene_bajada"].astype(bool)

transfo_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   tipo_transporte      5000 non-null   category      
 1   tiene_bajada         5000 non-null   bool          
 2   tiempo_subida        5000 non-null   datetime64[ns]
 3   tiempo_bajada        3795 non-null   datetime64[ns]
 4   tiempo_etapa         3795 non-null   float64       
 5   comuna_subida        4999 non-null   category      
 6   comuna_bajada        3786 non-null   category      
 7   parada_subida        4999 non-null   category      
 8   parada_bajada        3786 non-null   category      
 9   dist_ruta_paraderos  3795 non-null   float64       
 10  dist_eucl_paraderos  3795 non-null   float64       
 11  x_subida             3795 non-null   float64       
 12  y_subida             3795 non-null   float64       
 13  x_bajada             3795 non-nul

In [ ]:
def map_time_range(date):
    time = date.time()
    if (datetime.time(00, 00, 00) <= time) and (time <= datetime.time(5, 59, 59)):
        return "early"
    elif (datetime.time(6, 00, 00) <= time) and (time <= datetime.time(11, 59, 59)):
        return "morning"
    elif (datetime.time(12, 00, 00) <= time) and (time <= datetime.time(17, 59, 59)):
        return "afternoon"
    elif (datetime.time(18, 00, 00) <= time) and (time <= datetime.time(23, 59, 59)):
        return "night"

transfo_df["time_range"] = transfo_df["tiempo_subida"].map(map_time_range)

In [ ]:
def map_time_type(date):
    time = date.time()
    if (datetime.time(6, 00, 00) <= time) and (time <= datetime.time(8, 59, 59)):
        return "rush_time-morning"
    elif (datetime.time(17, 00, 00) <= time) and (time <= datetime.time(19, 59, 59)):
        return "rush_time-night"
    else:
        return "normal"

transfo_df["time_type"] = transfo_df["tiempo_subida"].map(map_time_type)
transfo_df.head()

,tipo_transporte,tiene_bajada,tiempo_subida,tiempo_bajada,tiempo_etapa,comuna_subida,comuna_bajada,parada_subida,parada_bajada,dist_ruta_paraderos,dist_eucl_paraderos,x_subida,y_subida,x_bajada,y_bajada,time_range,time_type
0,BUS,True,2025-04-21 08:48:04,2025-04-21 08:50:39,155.0,RECOLETA,RECOLETA,T-4-19-SN-40,E-4-19-SN-55,853.0,825.0,347201.0,6302489.0,347201.0,6302489.0,morning,rush_time-morning
1,BUS,True,2025-04-21 08:51:46,2025-04-21 08:54:58,192.0,RECOLETA,RECOLETA,E-4-19-SN-55,L-4-4-50-OP,1090.0,983.0,346625.0,6303299.0,346625.0,6303299.0,morning,rush_time-morning
2,BUS,True,2025-04-21 15:34:28,2025-04-21 15:39:01,273.0,RECOLETA,RECOLETA,L-4-12-20-PO,E-4-295-OP-5,1127.0,959.0,347168.0,6302522.0,347168.0,6302522.0,afternoon,normal
3,METRO,False,2025-04-21 17:27:55,NaT,NaN,LAS CONDES,NaN,LOS DOMINICOS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,afternoon,rush_time-night
4,METRO,True,2025-04-21 09:12:16,2025-04-21 09:38:40,1584.0,ESTACION CENTRAL,SANTIAGO,ESTACION CENTRAL,PLAZA DE ARMAS,6240.0,3064.0,346372.0,6299031.0,346372.0,6299031.0,morning,normal


# ¿Cómo varía el tiempo de viaje respecto la hora(hora normal y hora de punta)?

hora de punta: 6-9am 5-8pm

In [ ]:
df_has_bajada = transfo_df[transfo_df["tiene_bajada"] == True]
df_has_bajada.head()

,tipo_transporte,tiene_bajada,tiempo_subida,tiempo_bajada,tiempo_etapa,comuna_subida,comuna_bajada,parada_subida,parada_bajada,dist_ruta_paraderos,dist_eucl_paraderos,x_subida,y_subida,x_bajada,y_bajada,time_range,time_type
0,BUS,True,2025-04-21 08:48:04,2025-04-21 08:50:39,155.0,RECOLETA,RECOLETA,T-4-19-SN-40,E-4-19-SN-55,853.0,825.0,347201.0,6302489.0,347201.0,6302489.0,morning,rush_time-morning
1,BUS,True,2025-04-21 08:51:46,2025-04-21 08:54:58,192.0,RECOLETA,RECOLETA,E-4-19-SN-55,L-4-4-50-OP,1090.0,983.0,346625.0,6303299.0,346625.0,6303299.0,morning,rush_time-morning
2,BUS,True,2025-04-21 15:34:28,2025-04-21 15:39:01,273.0,RECOLETA,RECOLETA,L-4-12-20-PO,E-4-295-OP-5,1127.0,959.0,347168.0,6302522.0,347168.0,6302522.0,afternoon,normal
4,METRO,True,2025-04-21 09:12:16,2025-04-21 09:38:40,1584.0,ESTACION CENTRAL,SANTIAGO,ESTACION CENTRAL,PLAZA DE ARMAS,6240.0,3064.0,346372.0,6299031.0,346372.0,6299031.0,morning,normal
5,METRO,True,2025-04-21 10:49:35,2025-04-21 11:17:20,1665.0,SANTIAGO,LA FLORIDA,PLAZA DE ARMAS,BELLAVISTA DE LA FLORIDA,11630.0,10307.0,351430.0,6289929.0,351430.0,6289929.0,morning,normal


In [ ]:
df_has_bajada.groupby("time_type").agg({"tiempo_etapa": "mean"})

,tiempo_etapa
time_type,
normal,1167.508742
rush_time-morning,1340.028490
rush_time-night,1344.335397


In [ ]:
df_has_bajada.groupby(["time_type", "tipo_transporte"], observed=True).agg({"tiempo_etapa": "mean"})

tiempo_etapa
time_type         tipo_transporte              
normal            BUS                902.424514
                  METRO             1381.771305
                  METROTREN          881.147059
                  ZP                1086.783217
rush_time-morning BUS               1041.310440
                  METRO             1545.280265
                  METROTREN          880.500000
                  ZP                1230.300000
rush_time-night   BUS               1222.069767
                  METRO             1454.527629
                  METROTREN          829.375000
                  ZP                1133.262626

# ¿Cómo varían la comuna subida y comuna bajada respecto la hora?
(cómo circulan las gentes.)

hora: 
early [0-6)
morning [6-12)
afternoon [12-18)
night [18-24)


In [ ]:
pv_subida = pd.pivot_table(data=df_has_bajada, index='comuna_subida', columns="time_range", aggfunc="size", observed=False)
pv_bajada = pd.pivot_table(data=df_has_bajada, index='comuna_bajada', columns="time_range", aggfunc="size", observed=False)

In [ ]:
pv_subida.sort_values(by="early", ascending=False).head()[["early"]]

time_range,early
comuna_subida,
SANTIAGO,3
HUECHURABA,2
CERRILLOS,2
LAS CONDES,2
MAIPÚ,2


In [ ]:
pv_bajada.sort_values(by="early", ascending=False).head()[["early"]]

time_range,early
comuna_bajada,
LAS CONDES,4
MAIPÚ,2
HUECHURABA,2
SANTIAGO,2
ESTACION CENTRAL,1


In [ ]:
pv_subida.sort_values(by="morning", ascending=False).head()[["morning"]]

time_range,morning
comuna_subida,
SANTIAGO,171
LAS CONDES,129
PROVIDENCIA,110
LA FLORIDA,87
LO PRADO,86


In [ ]:
pv_bajada.sort_values(by="morning", ascending=False).head()[["morning"]]

time_range,morning
comuna_bajada,
SANTIAGO,301
PROVIDENCIA,217
LAS CONDES,205
MAIPÚ,58
MACUL,57


In [ ]:
pv_subida.sort_values(by="afternoon", ascending=False).head()[["afternoon"]]

time_range,afternoon
comuna_subida,
SANTIAGO,216
PROVIDENCIA,147
LAS CONDES,135
ÑUÑOA,49
MACUL,47


In [ ]:
pv_bajada.sort_values(by="afternoon", ascending=False).head()[["afternoon"]]

time_range,afternoon
comuna_bajada,
SANTIAGO,165
LAS CONDES,134
PROVIDENCIA,122
ÑUÑOA,59
LA FLORIDA,54


In [ ]:
pv_subida.sort_values(by="night", ascending=False).head()[["night"]]

time_range,night
comuna_subida,
SANTIAGO,158
PROVIDENCIA,149
LAS CONDES,124
ÑUÑOA,34
MAIPÚ,30


In [ ]:
pv_bajada.sort_values(by="night", ascending=False).head()[["night"]]

time_range,night
comuna_bajada,
SANTIAGO,99
PROVIDENCIA,74
LAS CONDES,73
NUNOA,50
LA FLORIDA,46


## ¿Qué variación de densidad de uso hay entre métodos de transporte (metro - bus) según el día (día laboral y fin de semana)?


# Graficar

heatmap con mapa de santiago